<a href="https://colab.research.google.com/github/mobadara/finbert-sentiment-analyzer-api/blob/main/notebooks/01_eda_and_data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **FINANCIAL NEWS SENTIMENT ANALYSIS**
## 📈 **EDA & Data Engineering**


**Author:** [Muyiwa J. Obadara](https://portfolio-frontend-livid.vercel.app)

**Project:** FinBERT Sentiment API Backend

### **Objective**
The goal of this notebook is to perform Exploratory Data Analysis (EDA) and preprocess the **[Financial PhraseBank](https://huggingface.co/datasets/FinanceMTEB/financial_phrasebank)** dataset. Financial text is highly specialized; words that represent positive sentiment in a general context might represent risk in a financial context.

By analyzing the class distributions and cleaning the text, we will create a highly optimized dataset ready to fine-tune the `ProsusAI/finbert` model in the next phase of this pipeline.

### **Dataset Details**
* **Source:** `financial_phrasebank` (via Hugging Face Datasets)
* **Configuration:** `sentences_allagree` (Using only sentences where 100% of human annotators agreed on the sentiment)
* **Labels:** 0 (Negative), 1 (Neutral), 2 (Positive)

### **Setup**

In [ ]:
from datasets import load_dataset
try:
    from google.colab import userdata
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
import shutil
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['blue'])

In [ ]:
os.makedirs('figures', exist_ok=True)
os.makedirs('datasets', exist_ok=True)

In [ ]:
dataset = load_dataset('FinanceMTEB/financial_phrasebank')

In [ ]:
dataset

The dataset has two splits:
1. Train Dataset contains 1264 instances.
2. Test Dataset contains 1000 instances.

We need to convert the train dataset to a pandas datafrome exploration.

In [ ]:
df_train = dataset['train'].to_pandas()
df_train.head()

### **Data Check and Cleaning**

In [ ]:
# Check for `null` values
nulls_per_column = df_train.isna().sum()

# Check for duplicated values
duplicated_instances = df_train.duplicated().sum()

print("Null Values Per Column:")
print(nulls_per_column)
print("-" * 30)
print(f"Number of Duplicated Instances: {duplicated_instances}")

We see that our dataset (in tabular form) contains three columns: **text**, **label_text** and **label**.

The **`label_text`** has three unique represents the sentiments of each text in human redable form, which are **positive**, **negative** or **neutral**.

Each of this is encoded in the `label` as follows:
label_text | code
-----------|-----
negative   | `0`
neutral    | `1`
positive   | `2`


We see from our result that one instance in the dataset is exactly repeated. We drop the duplicated values in the following cell.

In [ ]:
df_train.drop_duplicates(inplace=True)
assert df_train.duplicated().sum() == 0

Now that the duplicated entries are removed, we need to visualize our class distribution.

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_train,
              x='label_text',
              order=['negative', 'neutral', 'positive'])
plt.title('Distribution of Sentiments in Financial PhraseBank', fontsize=14)
plt.ylabel('Number of Sentences', fontsize=12)
plt.xlabel('Sentiment', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add percentages on top of each bar
total = len(df_train)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')

plt.savefig('figures/class_distribution.png')
plt.show()

Now that we understand our dataset and we have cleaned it, we need to save the cleaned dataset for future use.

In [ ]:
df_train.to_csv('datasets/cleaned_financial_phrasebank.csv', index=False)

The following cell will only run if woring in colab.

In [ ]:
if IN_COLAB:
    REPO_NAME = 'finbert-sentiment-analyzer-api'

    # Define paths
    SRC_CSV = 'datasets/cleaned_financial_phrasebank.csv'
    DEST_CSV = f'{REPO_NAME}/datasets/cleaned_financial_phrasebank.csv'
    SRC_IMG = 'figures/class_distribution.png'
    DEST_IMG = f'{REPO_NAME}/figures/class_distribution.png'

    # 1. Safely move/overwrite the CSV
    try:
        if os.path.exists(DEST_CSV):
            os.remove(DEST_CSV)
        shutil.move(SRC_CSV, DEST_CSV)
        print("  - CSV file successfully updated in repo.")
    except Exception as e:
        print(f"  - Error moving CSV: {e}")

    # 2. Safely move/overwrite the Figure
    try:
        if os.path.exists(DEST_IMG):
            os.remove(DEST_IMG)
        shutil.move(SRC_IMG, DEST_IMG)
        print("  - Figure successfully updated in repo.")
    except Exception as e:
        print(f"  - Error moving Figure: {e}")
else:
    print('Cell skipped (Not running in Google Colab environment).')

### **Add new files to the repositoy**

In [ ]:
if IN_COLAB:
    print("Executing Git commands using explicit repository targeting...")

    # The -C flag forces Git to operate strictly inside the cloned folder
    !git -C {REPO_NAME} status
    !git -C {REPO_NAME} add datasets/cleaned_financial_phrasebank.csv figures/class_distribution.png
    !git -C {REPO_NAME} commit -m "chore: update cleaned dataset and visualization figure"
    !git -C {REPO_NAME} pull --rebase origin main
    !git -C {REPO_NAME} push origin main

    print("\n✅ Done! Check your GitHub repository to see the updated files.")
else:
    print('Cell skipped (Not running in Google Colab environment).')

___
Muyiwa J. Obadara